# Pandas Series — Solved Exercises (Task 16)

This solutions notebook is a guided tour of the Pandas **Series**, the one-dimensional labelled array that is the building block of every DataFrame. It begins with the fundamentals — creating a Series, doing element-wise arithmetic on two of them, comparing them, and coercing their dtype — and then moves into two real datasets (IPL batting records and a restaurant menu) where the Series is used for genuine analysis: ranking, filtering, currency conversion, missing-value imputation, percentiles, and histograms.

The early exercises establish core behaviour. You create an empty Series with `pd.Series()` and learn why its default dtype is a moving target. You add, subtract, multiply, and divide two aligned Series, discovering that Pandas matches rows by **label**, not position, and that division silently produces floats while the other three operations stay integer. You compare two Series element-wise with `==`, `<`, and `>`, producing boolean Series that can later serve as masks. And you write a small function around `pd.to_numeric(..., errors='coerce')` that converts mixed-type data — numbers, strings, booleans — into a clean numeric Series, mapping anything unparseable to `NaN` instead of raising an error.

With the basics covered, Q-6 to Q-8 load `batsman_runs_series.csv` — every batter's career IPL runs from 2008 to 2022. You find the top-10 run-scorers with `sort_values(...).head(10)`, count how many players crossed 3,000 runs, and count how many exceeded the mean run tally, each time using a boolean mask and `.shape[0]`.

Q-9 and Q-10 tackle `items.csv`, a menu whose `item_price` is stored as text with a dollar sign (`$2.39`). You read it with `item_name` as the index and `squeeze` it into a Series, count the `NaN`s, strip the `$` and multiply by an exchange rate to get rupees, cast to `float`, and fill missing prices with the mean. The final questions compute the mean price, the 30th and 6th percentiles, plot a histogram with 50 bins, and count items priced between 1000 and 2000 rupees. Each answer is followed by its executed output so you can verify your logic. Read the explanation, then the code, then the output — in that order.

## 0. Setup

Import Pandas to begin.

In [ ]:
import pandas as pd

# 1. Series Basics and Arithmetic

The Series is Pandas' one-dimensional labelled array. This section covers creating one — empty or populated — and combining two of them with the four arithmetic operators.

### `Q-1:` Write a program to create an empty series.

## 1.1 The empty Series (Q-1)

`pd.Series()` returns `Series([], dtype: float64)` — an empty, one-dimensional container with no values and no explicit index. That is the whole answer to Q-1, but the accompanying `FutureWarning` is the interesting part: Pandas announces that in a future version the default dtype for an empty Series will become `object` instead of `float64`. Defaults changing over time is a recurring theme in Pandas, so the safe habit is to state your intent explicitly — `pd.Series(dtype='float64')` or `pd.Series(dtype='object')` — which both silences the warning and documents what you mean. An empty Series is genuinely useful as a placeholder accumulator in loops and as the neutral starting point for concatenation, so knowing how to create one deliberately (rather than accidentally) matters.

In [ ]:
# code here
pd.Series()

C:\Users\rajti\AppData\Local\Temp\ipykernel_19100\195155927.py:2: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  pd.Series()


Series([], dtype: float64)

### `Q-2:` Write a Pandas program to add, subtract, multiple and divide two Pandas Series.

## 1.2 Arithmetic between Series (Q-2)

Given `a = [2, 4, 6, 8, 10]` and `b = [1, 3, 5, 7, 10]`, the four operators work element-wise on **label-aligned** positions, not memory order: `a + b` gives `[3, 7, 11, 15, 20]`, `a - b` gives `[1, 1, 1, 1, 0]`, `a * b` gives `[2, 12, 30, 56, 100]`, and `a / b` gives `[2.0, 1.333..., 1.2, 1.142..., 1.0]`. Three outputs keep `dtype: int64`; division alone returns `float64`, because true division can produce fractions and Pandas will not silently truncate. Note also the **alignment rule**: if the two Series had different indexes, Pandas would match on the label and insert `NaN` wherever a label existed in one but not the other — arithmetic between Pandas objects is never a blind positional operation unless you drop to the underlying NumPy arrays.

In [ ]:
# code here
a = pd.Series([2, 4, 6, 8, 10])
b = pd.Series([1, 3, 5, 7, 10])

print(a+b)
print(a-b)
print(a*b)
print(a/b)

0     3
1     7
2    11
3    15
4    20
dtype: int64
0    1
1    1
2    1
3    1
4    0
dtype: int64
0      2
1     12
2     30
3     56
4    100
dtype: int64
0    2.000000
1    1.333333
2    1.200000
3    1.142857
4    1.000000
dtype: float64


# 2. Comparing Series (Q-3)

Comparison operators on a Series return a boolean Series of the same shape, comparing values element-wise (again by label).

### `Q-3` Write a Pandas program to compare the elements of the two Pandas Series.
Sample Series: [2, 4, 6, 8, 10], [1, 3, 5, 7, 10]



## 2.1 Boolean results from comparisons

For the same `a = [2, 4, 6, 8, 10]` and `b = [1, 3, 5, 7, 10]`, `a == b` is `[False, False, False, False, True]` (only the last pair, 10 and 10, matches), `a < b` is all `False` (every element of `a` is greater than or equal to its counterpart), and `a > b` is `[True, True, True, True, False]` — true everywhere except the tied final pair. The result dtype is `bool`. These boolean Series are not just trivia: they are exactly the **masks** used to filter DataFrames and Series later in this notebook (`data[data['batsman_run'] > 3000]`). Chaining them is also natural — combine masks with `&` (and), `|` (or), and `~` (not), remembering parentheses because comparisons bind tighter than the bitwise operators. So Q-3 is a rehearsal for the filtering questions that follow.

In [ ]:
# code here
a = pd.Series([2, 4, 6, 8, 10])
b = pd.Series([1, 3, 5, 7, 10])

print(a==b)
print(a<b)
print(a>b)

0    False
1    False
2    False
3    False
4     True
dtype: bool
0    False
1    False
2    False
3    False
4    False
dtype: bool
0     True
1     True
2     True
3     True
4    False
dtype: bool


# 3. Changing the Data Type (Q-5)

Real columns arrive mixed: numbers, text, and booleans jumbled together. Converting them safely is a routine first step in cleaning.

### `Q-5.`Write a function to change the data type of given a column or a Series. Function takes series and data type as input, returns the converted series.
```
series = pd.Series([1,2,'Python', 2.0, True, 100])
change to float type data
```
Note: Read about pd.to_numeric()

## 3.1 to_numeric with errors='coerce'

The task asks for a function that takes a Series and a target type and returns the converted Series. The given answer specialises to numeric using `pd.to_numeric(series, errors='coerce')`: a Series like `[1, 2, 'Python', 2.0, True, 100]` becomes `[1.0, 2.0, NaN, 2.0, 1.0, 100.0]` with dtype `float64`. Two things happen at once. First, every parseable element is converted — integers, floats, and even the boolean `True` (which counts as 1). Second, the unparseable `'Python'` does **not** raise; `errors='coerce'` replaces it with `NaN`, and the presence of that missing value forces the whole Series to `float64` (because integers cannot represent `NaN`). The alternative `errors='raise'` would throw a `ValueError`. This single call is the standard way to sanitise messy numeric-ish columns, and pairing it with `.fillna(...)` or `.dropna()` completes the cleaning workflow. For a version that also accepts the target dtype, you would add an `astype` step after this conversion.

In [ ]:
# code here
# Changing the dtype to numeric (int, float)
def change_dtype(series):
    return pd.to_numeric(series, errors = 'coerce')

series = pd.Series([1,2,'Python', 2.0, True, 100])
change_dtype(series)

0      1.0
1      2.0
2      NaN
3      2.0
4      1.0
5    100.0
dtype: float64

# 4. IPL Batsman Runs (Q-6 to Q-8)

The dataset `batsman_runs_series.csv` contains every IPL batter's career run tally from 2008 to 2022, with columns `batter` and `batsman_run`. The next three questions rank, threshold, and compare-against-the-mean.

Download data - https://drive.google.com/file/d/1LRhXwbEodeWXtzPhJCX0X9Lf_BECzvqb/view?usp=share_link
All Batsman runs series in IPL 2008 to 2022.

Below questions are based on this data.

## 4.1 Reading the data

`pd.read_csv('batsman_runs_series.csv')` loads the file into a DataFrame with one row per batter. Before analysing, glance at its structure — `shape` tells you how many players and columns, `head()` shows the first records, and `dtypes` confirms `batsman_run` arrived as a proper integer rather than text. This quick inspection is the habit that prevents later surprises: a run column stored as `object` would break sorting and arithmetic silently, and a stray index column from the CSV would add a phantom column. For a small exercise like this the file is clean, but on real data you would also check for duplicate batters (a player traded between franchises) and missing run values before trusting any count.

In [ ]:
data = pd.read_csv("batsman_runs_series.csv")

### `Q-6` Find top 10 most run getter from the series.

## 4.2 Top 10 run-getters (Q-6)

`data.sort_values('batsman_run', ascending=False).head(10)` sorts the whole frame by runs, largest first, and keeps the first ten rows. The result reveals the expected superstars — V Kohli (6634), S Dhawan (6244), DA Warner (5883), RG Sharma (5881), SK Raina (5536), AB de Villiers (5181), CH Gayle (4997), MS Dhoni (4978), and so on. Two idioms to note: `ascending=False` because we want the biggest first, and `.head(10)` to limit the display. The equivalent, often faster call is `data.nlargest(10, 'batsman_run')`, and if you only needed the names you would select that column first (`data.sort_values(...)['batter'].head(10)`). Ranking a column and slicing the top rows is the single most common 'leaderboard' pattern in Pandas.

In [ ]:
# code here
data.sort_values("batsman_run", ascending = False).head(10)

             batter  batsman_run
569         V Kohli         6634
462        S Dhawan         6244
130       DA Warner         5883
430       RG Sharma         5881
493        SK Raina         5536
27   AB de Villiers         5181
108        CH Gayle         4997
339        MS Dhoni         4978
452      RV Uthappa         4954
256      KD Karthik         4377

### `Q-7` No of players having runs above 3000

## 4.3 Threshold counts (Q-7)

`data[data['batsman_run'] > 3000].shape[0]` builds a boolean mask — `True` where a batter's runs exceed 3000 — filters the frame to just those rows, and `.shape[0]` reads off the row count. The answer is **20** players above 3000 runs. The pattern 'mask → filter → count rows' recurs constantly, and `.shape[0]` is the quickest equivalent to `len(filtered)` or `filtered.shape[0]`. Alternatives include `(data['batsman_run'] > 3000).sum()` — summing a boolean Series counts the `True`s directly — which avoids materialising the filtered frame and is marginally faster. Get comfortable reading `data[condition]` aloud as 'the rows of data where condition holds'; it is the foundation of all the filtering that follows.

In [ ]:
# code here
data[data["batsman_run"]>3000].shape[0]

20

### `Q-8` No of players having runs above mean value?

## 4.4 Above-average count (Q-8)

`data[data['batsman_run'] > data['batsman_run'].mean()].shape[0]` compares each batter's runs against the **mean** of the whole column and counts those above it — **128** players. This is a small but important lesson in distributions: the mean of a run column is dragged *down* by the long tail of players with few runs, so far more than half the players sit above the mean. (For heavily skewed data the median is often the more meaningful 'typical' value — the next dataset will use percentiles for exactly that reason.) Mechanically, `data['batsman_run'].mean()` returns a scalar that broadcasts against the column in the comparison, so no loop is needed. This 'compare a column to its own aggregate' pattern underlies percentile analysis, outlier flagging, and normalisation.

In [ ]:
# code here
data[data["batsman_run"] > data["batsman_run"].mean()].shape[0]

128

# 5. Restaurant Items — Cleaning Prices (Q-9)

The `items.csv` menu stores prices as text with a currency symbol, plus blanks. This section reads it into a Series, converts dollars to rupees, and imputes the missing prices.

Download data - https://drive.google.com/file/d/1QZuZ5bypUInfVvarHACLAi8tXXHvb8xd/view?usp=share_link

file name - items.csv



### `Q-9` 
    i. Read `items.csv` making `item_name` as index.
    ii. Show no of nan values
    ii. Item price is given in $, so convert it to rupees without currency symbol.
    iii. Make data type of newly made series as float.
    iv. Fill nan with mean of the series


How csv file looks

```
item_name	item_price
Chips and Fresh Tomato Salsa	$2.39
Izze	$3.39
Nantucket Nectar	$3.39
Chips and Tomatillo-Green Chili Salsa	$2.39
Chicken Bowl	$16.98

```

## 5.1 Read, convert, and fill

**Step i — read with a custom index.** `pd.read_csv('items.csv', index_col=['item_name'])` makes `item_name` the Series index instead of a data column, and `.squeeze('columns')` collapses the single remaining column (`item_price`) into a proper Series. Without the squeeze you would hold a one-column DataFrame; with it you hold a Series, which is the right shape for the subsequent `.apply`, `.mean`, and `.quantile` calls.

**Step ii — count missing values.** `items.isna().sum()` reports **50** nulls — a substantial chunk of the menu has no price. Counting before filling tells you how much of the column you are about to invent.

**Step ii — strip the currency and convert.** The helper `rupees(x)` slices off the leading `$` (`x[1:]`) and multiplies by an exchange rate (82.49), wrapped in a `try/except` because some entries are `NaN` (or otherwise not strings) and slicing them would raise. `items.apply(rupees)` runs this element-wise across the Series, turning `'$2.39'` into `197.1511` rupees. The `try/except` is doing the same job that `errors='coerce'` did earlier — letting malformed entries fall through rather than crash the whole conversion. In production you would often prefer `.str.replace('$','', regex=False).astype(float)` for the whole column at once, which is vectorised and clearer.

**Step iii — ensure float dtype.** Reassigning `items = items.apply(rupees)` materialises the converted values, and because `float(y)` was applied inside the helper the resulting Series is already `float64` — required so the later `NaN` imputation and percentile maths behave numerically.

**Step iv — impute.** `items.fillna(items.mean())` replaces every missing price with the mean of the known prices — a simple, defensible imputation for a small menu, though it does compress the variance and should be used knowingly. The cleaned Series is now complete and numeric, ready for the statistics of the next section.

In [ ]:
# code here
#1
items = pd.read_csv("items.csv", index_col = ['item_name']).squeeze("columns")
items

item_name
Chips and Fresh Tomato Salsa              $2.39 
Izze                                      $3.39 
Nantucket Nectar                          $3.39 
Chips and Tomatillo-Green Chili Salsa     $2.39 
Chicken Bowl                             $16.98 
                                          ...   
Steak Burrito                            $11.75 
Steak Burrito                            $11.75 
Chicken Salad Bowl                       $11.25 
Chicken Salad Bowl                        $8.75 
Chicken Salad Bowl                        $8.75 
Name: item_price, Length: 4622, dtype: object

In [ ]:
#2
items.isna().sum()

50

In [ ]:
#2ii
def rupees(x):
    try:
        y = x[1:]
    except:
        # Sometimes Dollar sign is not there which throws an exception 
        y = x
    return float(y)*82.49
items.apply(rupees)

item_name
Chips and Fresh Tomato Salsa              197.1511
Izze                                      279.6411
Nantucket Nectar                          279.6411
Chips and Tomatillo-Green Chili Salsa     197.1511
Chicken Bowl                             1400.6802
                                           ...    
Steak Burrito                             969.2575
Steak Burrito                             969.2575
Chicken Salad Bowl                        928.0125
Chicken Salad Bowl                        721.7875
Chicken Salad Bowl                        721.7875
Name: item_price, Length: 4622, dtype: float64

In [ ]:
#3
items = items.apply(rupees)

In [ ]:
#4
items.fillna(items.mean())

item_name
Chips and Fresh Tomato Salsa              197.1511
Izze                                      279.6411
Nantucket Nectar                          279.6411
Chips and Tomatillo-Green Chili Salsa     197.1511
Chicken Bowl                             1400.6802
                                           ...    
Steak Burrito                             969.2575
Steak Burrito                             969.2575
Chicken Salad Bowl                        928.0125
Chicken Salad Bowl                        721.7875
Chicken Salad Bowl                        721.7875
Name: item_price, Length: 4622, dtype: float64

# 6. Price Statistics and Visualisation (Q-10)

With a clean numeric Series in hand, the final questions compute summary statistics, plot its distribution, and count items in a price band.

### `Q-10`: 
    i. Find mean price
    ii. Find 30th and 6th percentile value
    iii. Plot Histogram on price with bin size 50
    iv. No of items price lies between [1000 to 2000]



## 6.1 Mean and percentiles

`items.mean()` returns **615.63** rupees — the average menu price after conversion. `items.quantile(q=0.3)` returns the **30th percentile** and `items.quantile(q=0.06)` the **6th percentile** — the price below which 30% (or 6%) of items fall; the notebook's executed output for the 6th percentile is 103.11 rupees. Percentiles are the robust alternative to the mean on skewed data: because a handful of expensive items (the 1400-rupee burrito bowls) inflate the average, the median and lower percentiles, which ignore that tail's magnitude, describe the 'typical' item far better. `quantile` accepts any fraction between 0 and 1, a list of fractions (`items.quantile([0.25, 0.5, 0.75])` gives the quartiles in one call), and interpolation methods for values between observed points. Reading the distribution through percentiles — not just the mean — is what separates careful analysis from a single misleading number.

In [ ]:
# code here
#1
items.mean()

615.6254681102482

In [ ]:
#2
items.quantile(q=0.3)
items.quantile(q=0.06)

103.1125

## 6.2 Histogram and price bands

`items.plot.hist(bins=50)` plots the distribution of prices in 50 bins — the Series plotting accessor, `plot`, is a thin wrapper over matplotlib that understands Pandas objects and labels the axes for you. Expect a strongly right-skewed histogram: most items cluster at the low end with a long tail of expensive dishes, which is precisely why the mean exceeded the low percentiles calculated above. A `bins` count controls the resolution of that picture (too few bins hides structure, too many adds noise).

The last query, `items[(items > 1000) & (items < 2000)]`, filters to items priced strictly between 1000 and 2000 rupees by combining two boolean masks with `&` — the parentheses are mandatory, because `&` binds more tightly than the comparison operators in Python's precedence table. The length of that filtered Series answers part iv. Taken together, mean, percentiles, histogram, and band-count paint a complete picture of the menu's price structure.

In [ ]:
#3
items.plot.hist(bins=50)

<AxesSubplot:ylabel='Frequency'>

<Figure size 640x480 with 1 Axes>

In [ ]:
items[(items>1000) & (items<2000)]

item_name
Chicken Bowl           1400.6802
Chicken Salad Bowl     1856.0250
Steak Burrito          1483.1702
Chicken Burrito        1443.5750
Chicken Bowl           1443.5750
                         ...    
Chicken Bowl           1856.0250
Steak Bowl             1938.5150
Chicken Bowl           1443.5750
Chips and Guacamole    1468.3220
Chicken Salad Bowl     1443.5750
Name: item_price, Length: 116, dtype: float64

## Summary

Task 16 walked the Pandas Series from first principles to real analysis. Q-1 created an empty Series and surfaced a dtype-default warning; Q-2 demonstrated label-aligned addition, subtraction, multiplication, and (float-producing) division; Q-3 produced boolean comparison Series that double as filters; and Q-5 converted messy mixed-type data with `pd.to_numeric(..., errors='coerce')`, turning bad entries into `NaN`. On the IPL runs data you ranked the top-10 scorers with `sort_values(...).head(10)`, counted 20 players above 3000 runs, and 128 above the mean using boolean masks and `.shape[0]`. On the menu data you read a CSV with a custom index and `.squeeze`, counted 50 missing prices, stripped `$` and converted to rupees via an `apply` with error handling, cast to `float`, and imputed nulls with the mean. The finale computed the mean price, the 30th and 6th percentiles with `quantile`, plotted a 50-bin histogram, and counted items in the 1000–2000 band using a `&`-combined mask. The toolkit: create, compute, filter, clean, summarise, visualise.